Libararies

In [149]:
import pulp
from pulp import LpProblem, LpMaximize, LpVariable, lpSum, LpBinary, PULP_CBC_CMD, value

In [150]:
import pandas as pd
import numpy as np
import os

import sys
from pathlib import Path
import importlib

PROJECT_ROOT = Path(r"C:\Users\jackg\code\F1Fantasy")
MODELS_DIR = PROJECT_ROOT / "notebooks" / "models"

sys.path.append(str(MODELS_DIR))

import teamOptimizer as to
importlib.reload(to)

<module 'teamOptimizer' from 'c:\\Users\\jackg\\code\\F1Fantasy\\notebooks\\models\\teamOptimizer.py'>

### File Import

In [151]:
working_directory = Path.cwd().parent.parent
print(working_directory)


#FIXME: consider not having these vals hardcoded or wrapping in a read function that
#  has the ability to do both hard coded or default most recent week

file_path_drivers = working_directory / "data" / "predictions" / "drivers" / "driver_predictions_2026.csv" 
file_path_constr = working_directory / "data" / "predictions" / "constructors" / "constructor_preds_2026.csv"
file_path_roster = working_directory / "data" / "myTeam" / "teamConfig.csv"

predicted_points_constr = pd.read_csv(file_path_constr).drop(columns=["Unnamed: 0"])
predicted_points_drivers = pd.read_csv(file_path_drivers)

last_week_roster = pd.read_csv(file_path_roster).drop(columns=["Unnamed: 0"])


c:\Users\jackg\code\F1Fantasy


### Filter the predicted points to the max race of the max year

In [152]:
year = max(predicted_points_drivers["year"])
race = max(predicted_points_drivers["race_num"])
print(year, race)

2026 6


In [153]:
predicted_points_constr = predicted_points_constr[(predicted_points_constr["year"] == year) & (predicted_points_constr["race_num"] == race)]
predicted_points_drivers = predicted_points_drivers[(predicted_points_drivers["year"] == year) & (predicted_points_drivers["race_num"] == race)]
#need this to be the prior week so 1- current week #FIXME: will this break when there is a 2 or three week gap between races?
last_week_roster = last_week_roster[(last_week_roster["race"] == race-1) & (last_week_roster["year"] == year)]


In [154]:
predicted_points_constr.head()

,year,race_num,constructor,price,predicted_points
33,2026,6,mer,30.2,81.409047
34,2026,6,mcl,28.6,70.285998
35,2026,6,fer,24.2,68.964515
36,2026,6,rbr,29.1,46.844220
37,2026,6,alp,14.3,26.979374


### Shrink Factor Towards the Mean

In [155]:
pred_mean = predicted_points_drivers["predicted_points"].mean()
predicted_points_drivers["predicted_points"] = (
    0.8 * predicted_points_drivers["predicted_points"] + 0.2 * pred_mean
)

In [156]:
predicted_points_drivers.sort_values("predicted_points", ascending=False).head(30)

,year,race_num,driver,price,constructor,predicted_points
67,2026,6,VER,28.2,RBR,28.213021
66,2026,6,RUS,28.3,MER,27.228907
70,2026,6,ANT,24.1,MER,24.936581
71,2026,6,LEC,23.7,FER,22.908537
72,2026,6,HAM,23.2,FER,22.169166
68,2026,6,NOR,26.5,MCL,19.918447
69,2026,6,PIA,24.6,MCL,19.463185
73,2026,6,HAD,13.3,RBR,10.004479
77,2026,6,BEA,9.2,HAS,9.264274
75,2026,6,SAI,12.4,WIL,9.100988


In [157]:
last_week_roster

,Race_team,year,race,constructor_1,constructor_2,driver_1,driver_2,driver_3,driver_4,driver_5,bonus_driver,team


In [158]:
last_week_roster.head()

,Race_team,year,race,constructor_1,constructor_2,driver_1,driver_2,driver_3,driver_4,driver_5,bonus_driver,team


In [159]:
last_week_lineup = {
    "drivers": {"RUS", "BEA", "HAD", "HUL", "LAW"}, 
    "constructors": {"MER", "RB"}
}

In [160]:
import inspect
print(inspect.signature(to.optimize_team))

(budget: float, drivers, cons, last_week_lineup: dict | None = None, free_transfers_avail: int = 2, points_col: str = 'predicted_points', n_drivers: int = 5, n_constructors: int = 2, max_drivers_per_team: int | None = None, use_drs: bool = False, drs_multiplier: float = 2.0, solver_msg: bool = False, require_driver_from_each_constructor: bool = False, min_drivers_per_selected_constructor: int = 1)


In [161]:
print(pulp.listSolvers(onlyAvailable=True))

['GLPK_CMD']


In [162]:
drivers_sel, cons_sel, summary = to.optimize_team(
    budget=104.3,
    drivers=predicted_points_drivers,
    cons=predicted_points_constr,
    last_week_lineup = last_week_lineup, 
    free_transfers_avail = 2,
    points_col="predicted_points",
    n_drivers=5,
    n_constructors=2,
    max_drivers_per_team=2,
    use_drs=True,
    drs_multiplier=2.0,
    solver_msg=False,
    require_driver_from_each_constructor=True,
    min_drivers_per_selected_constructor=1
)

print(summary)
display(drivers_sel)
display(cons_sel)

{'status': 1, 'budget': 104.3, 'points_column_used': 'predicted_points', 'total_price': 102.8, 'gross_points': 193.95, 'total_transfers': 1, 'free_transfers_avail': 2, 'paid_transfers': 0, 'transfer_penalty': 0, 'net_points': 193.95}


,year,race_num,driver,price,constructor,predicted_points,team_key,driver_key
0,2026,6,RUS,28.3,MER,27.228907,MER,RUS
1,2026,6,HAD,13.3,RBR,10.004479,RBR,HAD
2,2026,6,BEA,9.2,HAS,9.264274,HAS,BEA
3,2026,6,STR,6.2,AMR,6.609563,AMR,STR
4,2026,6,LAW,7.5,RB,6.598554,RB,LAW


,year,race_num,constructor,price,predicted_points,team_key
0,2026,6,mer,30.2,81.409047,MER
1,2026,6,rb,8.1,25.606062,RB
